# Colab Worker C: Track C - Learned Watermark Encoder & Controls
This notebook trains the Dynamic Band-Allocation Watermarking Network ($k=32$ bits, PSNR=40 dB).
- **Guaranteed Drive Persistence**: All model checkpoints are saved directly to Google Drive **after every single epoch**.
- **Auto-Resume**: If Colab disconnects, re-running any cell automatically resumes from the last completed epoch—zero progress is ever lost.
- **Self-Contained Merge 1**: Generates `fitted_g.json` directly from your uploaded `track_a_transfer.csv`.

In [ ]:
# 1. GPU Check & Persistent Drive Mount
!nvidia-smi
from google.colab import drive
import os, glob
drive.mount('/content/drive')

# Locate watermark_internship in MyDrive or via shortcut
SHARED_DIR = None
candidates = [
    '/content/drive/MyDrive/watermark_internship',
    *glob.glob('/content/drive/.shortcut-targets-by-id/*/watermark_internship'),
]
for c in candidates:
    if os.path.exists(c):
        SHARED_DIR = c
        break

if not SHARED_DIR:
    SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
    os.makedirs(SHARED_DIR, exist_ok=True)

print(f'✓ Persistent Drive folder ready at: {SHARED_DIR}')
!mkdir -p $SHARED_DIR/checkpoints $SHARED_DIR/results $SHARED_DIR/sweep_results

In [ ]:
# 2. Clone repo & install lightweight dependencies
# Note: Track C uses the analytical Fourier surrogate (no heavy VAE packages needed)
!rm -rf /content/repo
!git clone https://github.com/ashhyyyy-vis/Internship.git /content/repo
%cd /content/repo
# Ensure data_prep import fix is applied
!sed -i 's/from typing import List, Tuple$/from typing import List, Tuple, Optional/' data_prep.py 2>/dev/null || true
!pip install -q pandas scipy tqdm

In [ ]:
# 3. Setup Dataset Splits (Unpack from Drive or generate in 30s)
import os, glob
!mkdir -p /content/repo/data/processed

# Search for processed_splits.zip across all mounted Drive paths
zip_candidates = [
    f'{SHARED_DIR}/processed_splits.zip',
    '/content/processed_splits.zip',
    *glob.glob('/content/drive/**/processed_splits.zip', recursive=True),
    *glob.glob('/content/drive/.shortcut-targets-by-id/**/processed_splits.zip', recursive=True),
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if zip_path:
    print(f'Found {zip_path}! Extracting into repository...')
    !unzip -q -o "{zip_path}" -d /content/repo/data/
    !unzip -q -o "{zip_path}" -d /content/repo/data/processed/
else:
    print('Dataset zip not found in Drive. Preparing dataset directly in Colab (<30s)...')
    !sed -i 's/from typing import List, Tuple$/from typing import List, Tuple, Optional/' data_prep.py 2>/dev/null || true
    !python data_prep.py
    !cp /content/repo/data/processed_splits.zip $SHARED_DIR/ 2>/dev/null || true

# Mirror directory structure so all loaders find images
!mkdir -p /content/repo/data/processed/fit /content/repo/data/processed/test /content/repo/data/processed/train
!cp -rn /content/repo/data/fit/* /content/repo/data/processed/fit/ 2>/dev/null || true
!cp -rn /content/repo/data/test/* /content/repo/data/processed/test/ 2>/dev/null || true
!cp -rn /content/repo/data/train/* /content/repo/data/processed/train/ 2>/dev/null || true

n_fit = len(os.listdir('/content/repo/data/processed/fit')) if os.path.exists('/content/repo/data/processed/fit') else 0
print(f'✓ Dataset ready! Found {n_fit} images available for training.')

In [ ]:
# 4. Step 0 (Merge 1): Generate fitted_g.json from track_a_transfer.csv
# If you uploaded track_a_transfer.csv to /content, this fits the Wiener model and saves to Drive
import os
csv_path = None
for cand in ['/content/track_a_transfer.csv', f'{SHARED_DIR}/sweep_results/track_a_transfer.csv', f'{SHARED_DIR}/results/track_a_transfer.csv']:
    if os.path.exists(cand):
        csv_path = cand
        break

fitted_out = f'{SHARED_DIR}/results/fitted_g.json'
if csv_path:
    print(f'Fitting Wiener model from: {csv_path}')
    !python -m track_a.fit \
        --transfer_csv "{csv_path}" \
        --out_json "{fitted_out}" \
        --psnr 40.0
    !cp "{fitted_out}" $SHARED_DIR/sweep_results/fitted_g.json 2>/dev/null || true
    !cp "{fitted_out}" /content/fitted_g.json 2>/dev/null || true
    print(f'✓ fitted_g.json generated and saved to Drive at {fitted_out}!')
else:
    print('Notice: track_a_transfer.csv not found yet. Drag-and-drop it into /content if you want to regenerate.')

In [ ]:
# 5. Step 1: Sanity Overfit Test (100 images, identity noise)
!python -m track_c.train \
    --manifest data/manifests/fit_manifest.json \
    --images_dir data/processed/fit \
    --overfit_100 \
    --epochs 10 \
    --save_dir $SHARED_DIR/checkpoints/overfit_check
print('✓ Sanity overfit test completed!')

In [ ]:
# 6. Step 2: Train 3 Fixed-band Controls (Low, Mid, High)
# Checkpoints are saved straight to Google Drive after EVERY epoch!
!python -m track_c.train --fixed_regime low  --epochs 25 --save_dir $SHARED_DIR/checkpoints/control_low
!python -m track_c.train --fixed_regime mid  --epochs 25 --save_dir $SHARED_DIR/checkpoints/control_mid
!python -m track_c.train --fixed_regime high --epochs 25 --save_dir $SHARED_DIR/checkpoints/control_high
print('✓ All 3 fixed controls trained and saved in Google Drive!')

In [ ]:
# 7. Step 3 (Merge 1): Train Main Model with Spectral-Transfer Surrogate
# Checkpoints are saved straight to Google Drive after EVERY epoch!
surrogate_path = f'{SHARED_DIR}/results/fitted_g.json'
if not os.path.exists(surrogate_path):
    surrogate_path = f'{SHARED_DIR}/sweep_results/fitted_g.json'

!python -m track_c.train \
    --surrogate_json "{surrogate_path}" \
    --epochs 30 \
    --save_dir $SHARED_DIR/checkpoints/main_stage2_surrogate

print('✓ Main model with spectral surrogate trained and saved in Google Drive!')